# Connecting Python to SQL — Sakila Lab

This lab explores the **Sakila** database to identify customers active in both May and June 2005, and compares their rental activity between the two months.

## Step 1 — Establish a connection to the Sakila database

In [2]:
!pip install pymysql

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

DB_USER     = 'root'
DB_PASSWORD = '69696363Aasql7'
DB_HOST     = 'localhost'
DB_PORT     = '3306'
DB_NAME     = 'sakila'

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)
with engine.connect() as conn:
    print('Connection successful ✓')

Connection successful ✓


In [ ]:
#2. Write a Python function called `rentals_month` that retrieves rental data for a given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame. The function should take in three parameters:

	#- `engine`: an object representing the database connection engine to be used to establish a connection to the Sakila database.
	#- `month`: an integer representing the month for which rental data is to be retrieved.
	#- `year`: an integer representing the year for which rental data is to be retrieved.

	#The function should execute a SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, and return it as a pandas DataFrame.


In [5]:
def rentals_month(engine, month, year):
    query = """
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = %(month)s
      AND YEAR(rental_date) = %(year)s
    """
    df = pd.read_sql(query, engine, params={"month": month, "year": year})
    return df

In [ ]:
#3. "Develop a Python function called `rental_count_month` that takes the DataFrame provided by `rentals_month` as input along with the month and year and returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year. 
	#The function should also include the month and year as parameters and use them to name the new column according to the month and year, for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".


	#*Hint: Consider making use of pandas [groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)*

In [6]:
def rental_count_month(df, month, year):
    col_name = f'rentals_{month:02d}_{year}'
    result = df.groupby('customer_id').size().reset_index(name=col_name)
    return result

In [ ]:
#4. Create a Python function called `compare_rentals` that takes two DataFrames as input containing the number of rentals made by each customer in different months and years. 
#The function should return a combined DataFrame with a new 'difference' column, which is the difference between the number of rentals in the two months.

In [7]:


def compare_rentals(df1, df2):
    combined = df1.merge(df2, on ="customer_id", how ="outer").fillna(0)
    col1 = df1.columns[1]
    col2 = df2.columns[1]
    combined['difference'] = combined[col1] - combined[col2]
    return combined

In [9]:
#llamamos funciones
df_may_2005 = rentals_month(engine, 5, 2005)
df_june_2005 = rentals_month(engine, 6, 2005)

df_may_count = rental_count_month(df_may_2005, 5, 2005)
df_june_count = rental_count_month(df_june_2005, 6, 2005)

df_result = compare_rentals(df_may_count, df_june_count)
print(df_result.head())

   customer_id  rentals_05_2005  rentals_06_2005  difference
0            1              2.0              7.0        -5.0
1            2              1.0              1.0         0.0
2            3              2.0              4.0        -2.0
3            4              0.0              6.0        -6.0
4            5              3.0              5.0        -2.0
